Task 1:


Selected Raw Features:
I plan to use the following existing features because the may provide useful information about a client's circumstances and potential crisis risk.

| Feature | Reason for Including It | Preprocessing |
|---|---|---|
| `region` | Different regions may have different client needs and service patterns. | One-hot encoding |
| `age_at_screening` | Age may help identify patterns among the different age groups. | Median imputation and scaling |
| `housing_status` | Housing stability may be related to a client's risk of experiencing a crisis. | Most-frequent imputation and one-hot encoding |
| `prior_crisis_events_12mo` | Previous crisis events may help identify clients who could use additional support. | Median imputation and scaling |
| `phq2_score` | Depression screening scores may provide information about a client's mental health needs. | Median imputation and scaling |
| `substance_use_flag` | Substance use concerns may provide additional information about a client's support needs. | Most-frequent imputation and one-hot encoding |
| `primary_support_type` | The type of support a client has may help identify differences in their support network. | Most-frequent imputation and one-hot encoding |
| `days_since_last_contact` | A longer period without contact may indicate a need for additional outreach. | Median imputation and scaling |

Proposed Derived Features:
I also plan to create three additional features from the existing dataset. Having these features may provide the model with additional ways to recognize patterns beyond what the original columns offer.

| New Feature | Created From | Why It May Help |
|---|---|---|
| `has_prior_crisis` | `prior_crisis_events_12mo` | Identifies whether a client had at least one previous crisis event. |
| `age_group` | `age_at_screening` | Groups clients into age ranges to help identify patterns among different age groups. |
| `contact_gap_category` | `days_since_last_contact` | Groups the time since the client's last contact to help identify patterns related to longer gaps. |

Preprocessing Logic for Proposed Derived Features:
### has_prior_crisis
- Create a binary feature using prior_crisis_events_12mo
- Assign 1 if the client had one or more previous crisis events
- Assign 0 if the client had no previous crisis events
- Keep missing values as missing so they can be handled through the imputation process

### age_group
- Create age category by using age_at_screening
- Use the following age ranges:
    - 18–29: Young Adult
    - 30–44: Adult
    - 45–59: Middle-Aged Adult
    - 60–74: Older Adult
    - 75–89: Senior Adult
- Keep missing ages as missing and use the categorical imputation process

### contact_gap_category
- Create the category using days_since_last_contact
- Use the following ranges:
    - 0–90 days: Short Gap
    - 91–365 days: Medium Gap
    - 366–730 days: Long Gap
- A value of 0 will be included in the Short Gap category since it may represent contact on the same day as screening
- Values that are above 730 days will remain in the Long Gap category since longer periods without contact are possible
- Keep missing values as missing and handle them through categorical imputation
- Negative values will be investigated and treated as missing if confirmed to be data errors

Task 2:

Recreate the Week 2 data split results so we have something to compare the baseline results with Week 3 feature engineered

In [7]:
import pandas as pd
from sklearn.model_selection import train_test_split

from week_2_community_health_eda import categorical_pipeline

#Load Dataset
df = pd.read_csv("community_risk_screening_train.csv")

#Select the Eight Screening Features
feature_columns = [
    "age_at_screening",
    "region",
    "housing_status",
    "prior_crisis_events_12mo",
    "phq2_score",
    "substance_use_flag",
    "primary_support_type",
    "days_since_last_contact"
]

#Separate Features and Target
x = df[feature_columns]
y = df["crisis_within_90_days"]

#Split Data into Training, Validation, and Test Sets
x_train, x_temp, y_train, y_temp = train_test_split(x, y, test_size = 0.30, random_state = 42, stratify=y)

x_val, x_test, y_val, y_test = train_test_split(
    x_temp, y_temp, test_size = 0.50, random_state = 42, stratify = y_temp
)

#Display Dataset
print("Training Records:", len(x_train))
print("Validation Records:", len(x_val))
print("Test Records:", len(x_test))

Training Records: 1680
Validation Records: 360
Test Records: 360


Create the three derived features:
- has_prior_crisis: Indicates whether the client had a previous crisis
- age_group: Groups the clients by their age
- contact_gap_category: Groups clients by the number of days since their last contact

In [10]:
import numpy as np

#Create Functions for Derived Features
def add_derived_features(data):

    #Make a Copy of Original Data
    data = data.copy()

    #Feature 1: has_prior_crisis
    data["has_prior_crisis"] = np.where(
        data["prior_crisis_events_12mo"].isna(), np.nan, (data["prior_crisis_events_12mo"] >= 1).astype(int)
    )

    #Feature 2: age_group
    data["age_group"] = pd.cut(
        data["age_at_screening"], bins = [17, 29, 44, 59, 74, 89], labels = ["Young Adult", "Adult", "Middle-Aged Adult", "Older Adult", "Senior Adult"]
    )

    #Feature 3: contact_gap_category
    data["contact_gap_category"] = pd.cut(
        data["days_since_last_contact"], bins = [-1, 90, 365, np.inf], labels = ["Short Gap", "Medium Gap", "Long Gap"]
    )

    return data

#Testing Only Using Training Data
#Add Derived Features to Training Data
x_train_engineered = add_derived_features(x_train)

#Display the New Features
print(x_train_engineered[
    ["has_prior_crisis", "age_group", "contact_gap_category"]
].head())

#Check the New Feature Columns
print("\nNew Feature Names:")
print(x_train_engineered.columns[-3:].tolist())

      has_prior_crisis     age_group contact_gap_category
2355               1.0   Older Adult           Medium Gap
1681               1.0   Young Adult             Long Gap
772                NaN  Senior Adult           Medium Gap
154                1.0   Young Adult             Long Gap
620                1.0   Young Adult            Short Gap

New Feature Names:
['has_prior_crisis', 'age_group', 'contact_gap_category']


In [13]:
#Checking Ages Fall Outside the Defined 18 - 89 Range and if There any Negative Contact Gaps

#Check Values Outside Expected Ranges
print("Minimum Age:", x_train["age_at_screening"].min())
print("Maximum Age:", x_train["age_at_screening"].max())

print("\nNegative Contact Gap Values:")
print((x_train["days_since_last_contact"] < 0).sum())

print("\nMissing Age Groups:")
print(x_train_engineered["age_group"].isna().sum())

#Check Missing Age Values
print("\nMissing Original Age Values:")
print(x_train["age_at_screening"].isna().sum())

#Check Missing Age Group Vales
print("\nMissing Derived Age Groups:")
print(x_train_engineered["age_group"].isna().sum())

Minimum Age: 18.0
Maximum Age: 89.0

Negative Contact Gap Values:
0

Missing Age Groups:
120

Missing Original Age Values:
120

Missing Derived Age Groups:
120


Derived Feature Validation Results:
- The minimum and maximum ages are withing the expect range of 18 - 89
- There is no negative contact gaps that were found
- All 120 missing age groups come from the 120 missing original age values, meaning we aren't making up any values or introducing any additional missing values

Separate Numerical and Categorical Features:

Separating original and derived features into numerical and categorical groups so they can be processed appropriately.

- Numerical Features: Contains the numerical values and will use median imputation to handle missing values and standard scaling
- Categorical Features: Contain categories or labels and will use most-frequent imputation and one-hot encoding

Once separated it allows the preprocessing pipeline to apply the correct transformations to each group.

In [15]:
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler

#Numerical Features
numeric_features = [
    "age_at_screening",
    "prior_crisis_events_12mo",
    "phq2_score",
    "days_since_last_contact",
    "has_prior_crisis"
]

#Categorical Features
categorical_features = [
    "region",
    "housing_status",
    "substance_use_flag",
    "primary_support_type",
    "age_group",
    "contact_gap_category"
]

#Display Feature Groups
print("Numerical Features:", numeric_features)
print("\nCategorical Features:", categorical_features)

Numerical Features: ['age_at_screening', 'prior_crisis_events_12mo', 'phq2_score', 'days_since_last_contact', 'has_prior_crisis']

Categorical Features: ['region', 'housing_status', 'substance_use_flag', 'primary_support_type', 'age_group', 'contact_gap_category']


In [25]:
#Create Numerical Preprocessing Pipeline
numeric_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy = "median")),
    ("scaler", StandardScaler())
])

#Create Categorical Preprocessing Pipeline
categorical_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy = "most_frequent")),
    ("encoder", OneHotEncoder(handle_unknown = "ignore"))
])

#Combine Numerical and Categorical Preprocessing
preprocessor = ColumnTransformer([
    ("numeric", numeric_pipeline, numeric_features),
    ("categorical", categorical_pipeline, categorical_features)
])

#Import Custom Transformer Tools
from sklearn.base import BaseEstimator, TransformerMixin

#Create Transformer for Derived Features
class FeatureEngineer(BaseEstimator, TransformerMixin):
    def fit(self, x, y=None):
        return self
    def transform(self, x):
        return add_derived_features(x)

#Create the Complete Machine Learning Pipeline
from sklearn.linear_model import LogisticRegression
full_pipeline = Pipeline([
    ("feature_engineering", FeatureEngineer()),
    ("preprocessor", preprocessor),
    ("classifier", LogisticRegression(
        max_iter = 1000,
        class_weight = "balanced",
        random_state = 42
    ))
])

#Training tje Complete Pipeline Using Training Data
full_pipeline.fit(x_train, y_train)

#Predict Outcomes Using Validation Data
y_val_predictions = full_pipeline.predict(x_val)

#Import Evaluation Metrics
from sklearn.metrics import recall_score, precision_score, f1_score

#Calculate Validation Metrics
recall = recall_score(y_val, y_val_predictions, pos_label = "crisis")
precision = precision_score(y_val, y_val_predictions, pos_label = "crisis")
f1 = f1_score(y_val, y_val_predictions, pos_label = "crisis")

#Display Results
print("Feature-Engineered Model Results:")
print("Recall:", round(recall, 3))
print("Precision:", round(precision, 3))
print("F1 Score:", round(f1, 3))

Feature-Engineered Model Results:
Recall: 0.774
Precision: 0.488
F1 Score: 0.599


Model Comparison Between Week 2 and 3:

| Metric | Week 2 | Week 3 | Difference |
|---|---:|---:|---:|
| Recall | 0.774 | 0.774 | No change |
| Precision | 0.512 | 0.488 | -0.024 |
| F1 Score | 0.617 | 0.599 | -0.018 |

The feature-engineered model in week 3 achieved a recall of 0.774, precision of 0.488, and F1 score of 0.599. Coming this to the logistic regression model from week 2, the recall remained the same, while the precision and F1 decreased slightly.

This shows us that adding the three derived features did not help improve the overall validation performance. But the features can still provide useful information so further testing is needed.

Verifying Derived Features in the Model:
Checking whether all three derived features were successfully included in the preprocessing pipeline and transformed into features that the logistic regression model can use.

In [26]:
#Getting the Features Names After Preprocessing
transformed_features = full_pipeline.named_steps["preprocessor"].get_feature_names_out()

#Check Whether the Three Derived Features Were Included
derived_features = [
    "has_prior_crisis", "age_group", "contact_gap_category"]

print("Checking Derived Features in the Model:")

for feature in derived_features:
    matches = [name for name in transformed_features if feature in name]
    print(f"{feature}: {len(matches)} transformed column(s)")
    print(matches)

Checking Derived Features in the Model:
has_prior_crisis: 1 transformed column(s)
['numeric__has_prior_crisis']
age_group: 5 transformed column(s)
['categorical__age_group_Adult', 'categorical__age_group_Middle-Aged Adult', 'categorical__age_group_Older Adult', 'categorical__age_group_Senior Adult', 'categorical__age_group_Young Adult']
contact_gap_category: 3 transformed column(s)
['categorical__contact_gap_category_Long Gap', 'categorical__contact_gap_category_Medium Gap', 'categorical__contact_gap_category_Short Gap']


Derived Feature Verification Results:
All the three derived features were successfully included in the preprocessing pipeline and transformed for the logistic regression model.

- has_prior_crisis produced 1 numerical feature
- age_group produced 5 encoded categorical features
- contact_gap_category produced 3 encoded categorical features

Totaling to 9 transformed columns that were created from the three derived features.

Data Leakage Check: Excluded Features:

Before training the model, I reviewed the dataset columns to identify features that could cause data leakage. The outreach_followup_tier column was excluded because it is assigned after the screening and may contain information about subsequent contacts. The crisis_within_90_days column was also excluded from the input features because it is the target being predicted. The client_screening_id column was excluded because it is only an identifier and does not provide useful predictive information.

In [31]:
#Features that must not be used as Model Inputs
excluded_features = [
    "client_screening_id",
    "outreach_followup_tier",
    "crisis_within_90_days",
    "coordinator_intake_notes"
]

#Check the Selected Model Features
print("Checking for Excluded Features:")

for feature in excluded_features:
    if feature in feature_columns:
        print(f"WARNING: {feature} is included!")
    else:
        print(f"PASS: {feature} is excluded.")

#Confirm that Model uses only the Selected Features
print("\nSelected Model Features:")
print(feature_columns)

#Confirm the Training Data uses the Selected Features
print("\nTraining Data Columns:")
print(list(x_train.columns))

Checking for Excluded Features:
PASS: client_screening_id is excluded.
PASS: outreach_followup_tier is excluded.
PASS: crisis_within_90_days is excluded.
PASS: coordinator_intake_notes is excluded.

Selected Model Features:
['age_at_screening', 'region', 'housing_status', 'prior_crisis_events_12mo', 'phq2_score', 'substance_use_flag', 'primary_support_type', 'days_since_last_contact']

Training Data Columns:
['age_at_screening', 'region', 'housing_status', 'prior_crisis_events_12mo', 'phq2_score', 'substance_use_flag', 'primary_support_type', 'days_since_last_contact']


Data Leakage Check Results:
The leakage check confirmed that all four excluded columns were successfully kept out of the model inputs. The model uses only the eight selected features that are available at the time of screening. This way it helps prevent the model from using future outcome information or unrelated identifiers when making predictions.

In [37]:
#Get the Fitted Numerical Preprocessing Pipeline
fitted_numeric_pipeline = full_pipeline.named_steps["preprocessor"].named_transformers_["numeric"]

#Get the Number of Records used to Fit the Scaler
training_records_used = fitted_numeric_pipeline.named_steps["scaler"].n_samples_seen_

#Display the Results
print("Training-Only Preprocessing Check")
print("Training Records:", len(x_train))
print("Records Used to Fit Scaler:", training_records_used)

#Verify the Results
if training_records_used == len(x_train):
    print("PASS: Scaler was fitted using the training records only.")
else:
    print("CHECK: Record counts do not match.")

Training-Only Preprocessing Check
Training Records: 1680
Records Used to Fit Scaler: 1680.0
PASS: Scaler was fitted using the training records only.


Training-Only Preprocessing Verification:

The training-only preprocessing check confirmed that the numerical scaler was fitted using 1,680 training records. This matches the size of the training dataset. While the preprocessing pipeline was fitted using the training data only, helping prevent information from the validation and test sets from influencing the model during training.

Derived Feature Leakage Check:

I have reviewed the three derived features to confirm that they use the information that is available to us at the time of screening.
- has_prior_crisis uses the number of crisis events from the previous 12 months
- age_group uses the client's age at screening
- contact_gap_category uses the number of days since the last recorded contact

None of these features requires any future outcome information, meaning they are appropriate to use fot the model inputs.

## Data Leakage Check Report:

The leakage checks confirmed that the model uses only information available at the time of screening.

### Selected Raw Features:
The model uses the following eight original features:
- age_at_screening
- region
- housing_status
- prior_crisis_events_12mo
- phq2_score
- substance_use_flag
- primary_support_type
- days_since_last_contact

### Excluded Features:
These following columns were excluded:
- crisis_within_90_days - This is the target variable were are trying to predict
- client_screening_id - This is an identifier only
- outreach_followup_tier - Contains some information that were assigned after screening and may include subsequent contacts
- coordinator_intake_notes - A place to take notes not used in the model

### Training-Only Preprocessing
The preprocessing pipeline was trained using only the 1,680 training records that was given to us. This means the imputer, scaler, and encoder learned from the training data without using information from the validation or test sets.

### Derived Feature Verification
The three derived features that were created and use the information that was available at screening:
- has_prior_crisis - The previous 12 month crisis events
- age_group - Age at screening
- contact_gap_category - Days since the last recorded contact

### Conclusion
This leakage checks confirmed that the model excludes any post screening information and only use the selected screening features.

In [34]:
import joblib

#Save the Trained Pipeline
joblib.dump(
    full_pipeline,
    "community_health_risk_pipeline.joblib"
)

print("Pipeline Saved Successfully!")

Pipeline Saved Successfully!


In [42]:
import joblib

#Load the Saved Pipeline
loaded_pipeline = joblib.load(
    "community_health_risk_pipeline.joblib"
)

print("Pipeline Loaded Successfully!")

#Make Predictions Using the Loaded Pipeline
saved_predictions = loaded_pipeline.predict(x_val)

#Display a Few Predictions
print("\nSample Predictions:")
print(saved_predictions[:10])

print("Number of Predictions:", len(saved_predictions))

Pipeline Loaded Successfully!

Sample Predictions:
['no_crisis' 'no_crisis' 'no_crisis' 'no_crisis' 'no_crisis' 'no_crisis'
 'no_crisis' 'no_crisis' 'no_crisis' 'no_crisis']
Number of Predictions: 360


Saved Pipeline Verification

I loaded the save pipeline successfully by using Joblib and test it with the validation dataset. The pipeline generated 360 predictions, matching the number of validation records. This confirms that the saved pipeline can be loaded and used to make predictions.

Task 3:

Analyzing Logistic Regression Feature Coefficients:

Examining the logistic regression coefficients to determine which of the three derived features has the largest absolute coefficient.

In [43]:
#Get Transformed Feature Names
feature_names = full_pipeline.named_steps["preprocessor"].get_feature_names_out()

#Get Logistic Regression Coefficients
coefficients = full_pipeline.named_steps["classifier"].coef_[0]

#Create a DataFrame of Feature Coefficients
coefficient_df = pd.DataFrame({
    "Feature": feature_names, "Coefficient": coefficients})

#Calculate the Absolute Coefficients
coefficient_df["Absolute Coefficient"] = (coefficient_df["Coefficient"].abs())

#Filter the Three Derived Features
derived_coefficients = coefficient_df[
    coefficient_df["Feature"].str.contains(
        "has_prior_crisis|age_group|contact_gap_category"
    )
]

#Sort by Largest
derived_coefficients = derived_coefficients.sort_values(
    by = "Absolute Coefficient",
    ascending = False
)

#Display the Results
print("Derived Feature Coefficients:")
print(derived_coefficients.to_string(index = False))

Derived Feature Coefficients:
                                     Feature  Coefficient  Absolute Coefficient
 categorical__contact_gap_category_Short Gap    -0.455614              0.455614
          categorical__age_group_Young Adult    -0.353554              0.353554
categorical__contact_gap_category_Medium Gap     0.201394              0.201394
         categorical__age_group_Senior Adult    -0.173273              0.173273
                categorical__age_group_Adult     0.153548              0.153548
  categorical__contact_gap_category_Long Gap     0.148409              0.148409
    categorical__age_group_Middle-Aged Adult     0.138013              0.138013
          categorical__age_group_Older Adult     0.129456              0.129456
                   numeric__has_prior_crisis    -0.096152              0.096152


Logistic Regression Coefficient Analysis:

What we got from the coefficient analysis is that contact_gap_category_Short Gap had the largest absolute coefficient with 0.456. The age_group_Young Adult category had the second largest absolute coefficient with 0.354, while has_prior_crisis had the smallest coefficient with 0.096. These results show us that the contact gap and age group features have a stronger coefficient values compare to the previous crisis indicator.

In [44]:
#Get all Transformed Features Names from Model
feature_names = full_pipeline.named_steps["preprocessor"].get_feature_names_out()

#Check for Both Contact Gap Features
print("Checking for Overlapping Features:")
for feature in feature_names:
    if "days_since_last_contact" in feature or "contact_gap_category_Short Gap" in feature:
        print(feature)

Checking for Overlapping Features:
numeric__days_since_last_contact
categorical__contact_gap_category_Short Gap


Checking for Overlapping Features:
The feature check confirmed that both days_since_last_contact and contact_gap_category_Short Gap are included in the logistic regression model. These features carry overlap information since both describe the time since the client's previous contact. The original feature records the exact number of days, while the derived feature groups the number of days into categories. However, I have not yet confirmed whether this overlap suppresses either feature or negatively affects model performance.

Testing Model Without days_since_last_contact

In [45]:
#Remove days_since_last_contact from the Numeric Features
test_numeric_features = [
    feature for feature in numeric_features
    if feature != "days_since_last_contact"
]

#Keep all Categorical Features, Including contact_gap_category
test_categorical_features = categorical_features.copy()

#Create a New Preprocessor
test_preprocessor = ColumnTransformer([
    ("numeric", numeric_pipeline, test_numeric_features),
    ("categorical", categorical_pipeline, test_categorical_features)
])

#Create New Pipeline
test_pipeline = Pipeline([
    ("feature_engineering", FeatureEngineer()),
    ("preprocessor", test_preprocessor),
    ("classifier", LogisticRegression(
        max_iter=1000,
        class_weight="balanced",
        random_state=42
    ))
])

#Train the Model Without days_since_last_contact
test_pipeline.fit(x_train, y_train)

#Make Validation Predictions
test_predictions = test_pipeline.predict(x_val)

#Calculate Validation Metrics
test_recall = recall_score(y_val, test_predictions, pos_label="crisis")
test_precision = precision_score(y_val, test_predictions, pos_label="crisis")
test_f1 = f1_score(y_val, test_predictions, pos_label="crisis")

#Display Results
print("Model Without days_since_last_contact")
print("Recall:", round(test_recall, 3))
print("Precision:", round(test_precision, 3))
print("F1 Score:", round(test_f1, 3))

Model Without days_since_last_contact
Recall: 0.755
Precision: 0.482
F1 Score: 0.588


Feature Group Impact Analysis:

To understand how much each feature group is contributing to the model performance, by removing one group at a time  and comparing the results with the full model. I will be using recall, precision, and F1 score to see which feature groups can help the model make better predictions.

In [46]:
#Define the Feature Groups\
feature_groups = {
    "Demographics": {
        "numeric": ["age_at_screening"],
        "categorical": ["age_group"]
    },
    "Prior Crisis History": {
        "numeric": ["prior_crisis_events_12mo", "has_prior_crisis"],
        "categorical": []
    },
    "Contact History": {
        "numeric": ["days_since_last_contact"],
        "categorical": ["contact_gap_category"]
    },
    "Screening and Support": {
        "numeric": ["phq2_score"],
        "categorical": [
            "region",
            "housing_status",
            "substance_use_flag",
            "primary_support_type"
        ]
    }
}

print("Feature Groups:")
for group, features in feature_groups.items():
    print(f"\n{group}:")
    print("Numeric:", features["numeric"])
    print("Categorical:", features["categorical"])

Feature Groups:

Demographics:
Numeric: ['age_at_screening']
Categorical: ['age_group']

Prior Crisis History:
Numeric: ['prior_crisis_events_12mo', 'has_prior_crisis']
Categorical: []

Contact History:
Numeric: ['days_since_last_contact']
Categorical: ['contact_gap_category']

Screening and Support:
Numeric: ['phq2_score']
Categorical: ['region', 'housing_status', 'substance_use_flag', 'primary_support_type']


In [50]:
#Calculate the Full Model's Validation Performance
full_predictions = full_pipeline.predict(x_val)

full_recall = recall_score(y_val, full_predictions, pos_label="crisis")
full_precision = precision_score(y_val, full_predictions, pos_label="crisis")
full_f1 = f1_score(y_val, full_predictions, pos_label="crisis")

#Store the Full Model Results
ablation_results = [{"Model": "Full Model", "Recall": full_recall, "Precision": full_precision, "F1 Score": full_f1}]

#Remove One Feature Group at a Time
for group_name, group_features in feature_groups.items():

    #Keep the Remaining Numeric Features
    remaining_numeric = [feature for feature in numeric_features if feature not in group_features["numeric"]]

    #Keep the Remaining Categorical Features
    remaining_categorical = [feature for feature in categorical_features if feature not in group_features["categorical"]]

    #Create a New Preprocessor
    group_preprocessor = ColumnTransformer([("numeric", numeric_pipeline, remaining_numeric), ("categorical", categorical_pipeline, remaining_categorical)])

    #Create a New Model Pipeline
    group_pipeline = Pipeline([
        ("feature_engineering", FeatureEngineer()),
        ("preprocessor", group_preprocessor),
        ("classifier", LogisticRegression(
            max_iter=1000,
            class_weight="balanced",
            random_state=42
    ))
])

    #Train the Model
    group_pipeline.fit(x_train, y_train)

    #Make Validation Predictions
    group_predictions = group_pipeline.predict(x_val)

    #Calculate Performance Metrics
    group_recall = recall_score(y_val, group_predictions, pos_label="crisis")
    group_precision = precision_score(y_val, group_predictions, pos_label="crisis")
    group_f1 = f1_score(y_val, group_predictions, pos_label="crisis")

    #Save the Results
    ablation_results.append({"Model": f"Without {group_name}", "Recall": group_recall, "Precision": group_precision, "F1 Score": group_f1})

#Display the Comparison
ablation_df = pd.DataFrame(ablation_results)
print("Feature Group Ablation Results:")
print(ablation_df.round(3).to_string(index = False))

Feature Group Ablation Results:
                        Model  Recall  Precision  F1 Score
                   Full Model   0.774      0.488     0.599
         Without Demographics   0.755      0.482     0.588
 Without Prior Crisis History   0.774      0.427     0.550
      Without Contact History   0.774      0.466     0.582
Without Screening and Support   0.792      0.362     0.497


Feature Group Impact Analysis Results:

The feature group comparison showed us that the screening and support features had the largest impact on the model performance. By removing this group it lowered the F1 score from 0.599 to 0.497. The previous crisis information was also useful to the model because when I remove the features, the F1 score dropped from 0.599 to 0.550. This means that the model performed worse when it no longer had information about the client's previous crisis events. By also removing the contact history it caused the F1 score to drop slightly to 0.582 and by removing the demographic features it had a smaller effect compare with the others, with an F1 score dropping to 0.588. These results show us that the screening and support features have the biggest impact on the model following by previous crisis history. Contact history and demographic features also helped but their impact is smaller. This shows us that the model relies more on the client's screening information, support needs and previous crisis experience when making the predictions.

In [51]:
#Get the Full F1 Score Model
baseline_f1 = ablation_df.loc[ablation_df["Model"] == "Full Model", "F1 Score"].iloc[0]

#Calculate F1 Score Changes
ablation_df["F1 Score Change"] = (ablation_df["F1 Score"] - baseline_f1)

#Display Results
print("Feature Group Impact Compared to Full Model:")
print(ablation_df.round(3).to_string(index=False))

Feature Group Impact Compared to Full Model:
                        Model  Recall  Precision  F1 Score  F1 Score Change
                   Full Model   0.774      0.488     0.599            0.000
         Without Demographics   0.755      0.482     0.588           -0.010
 Without Prior Crisis History   0.774      0.427     0.550           -0.048
      Without Contact History   0.774      0.466     0.582           -0.017
Without Screening and Support   0.792      0.362     0.497           -0.101


Feature Group Performance Impact:

The F1 score comparison showed how much each feature group contributed to the model performance. By removing the screening and support features caused the largest decrease in the F1 score by 0.101 and by removing the previous crisis history cause the second largest decrease of 0.048. However, removing the contact history caused a smaller decrease of 0.017, and while removing demographic features had the smallest impact with a decrease of 0.010. These results suggest that screening and support features and previous crisis history are the most important feature groups for the model to use. Contact history and demographic features also contributed, but they have a smaller impact on the model.

Individual Feature Importance Analysis:

I will examine how individual features will affect the model performance by removing one feature at a time and comparing the results with the full model. This will help identify which features provide useful information and which features may have a smaller impact on predictions.

In [52]:
#List All Numeric and Categorical Features
all_features = numeric_features + categorical_features

#Display the Features
print("Individual Features to Test:")
for feature in all_features: print(feature)
print("\nTotal Features:", len(all_features))

Individual Features to Test:
age_at_screening
prior_crisis_events_12mo
phq2_score
days_since_last_contact
has_prior_crisis
region
housing_status
substance_use_flag
primary_support_type
age_group
contact_gap_category

Total Features: 11


In [54]:
#Store the Full Model's Performance as the Starting Point
full_model_f1 = f1_score(y_val, full_pipeline.predict(x_val), pos_label="crisis")

#Features to Remove and Test
features_to_remove = {
        "age_at_screening": {
        "numeric": ["age_at_screening"],
        "categorical": ["age_group"]
    },
    "prior_crisis_events_12mo": {
        "numeric": ["prior_crisis_events_12mo", "has_prior_crisis"],
        "categorical": []
    },
    "phq2_score": {
        "numeric": ["phq2_score"],
        "categorical": []
    },
    "days_since_last_contact": {
        "numeric": ["days_since_last_contact"],
        "categorical": ["contact_gap_category"]
    },
    "region": {
        "numeric": [],
        "categorical": ["region"]
    },
    "housing_status": {
        "numeric": [],
        "categorical": ["housing_status"]
    },
    "substance_use_flag": {
        "numeric": [],
        "categorical": ["substance_use_flag"]
    },
    "primary_support_type": {
        "numeric": [],
        "categorical": ["primary_support_type"]
    }
}

#Store the Results
individual_results = []

#Test Removing Each Feature
for feature, columns in features_to_remove.items():
    remaining_numeric = [f for f in numeric_features if f not in columns["numeric"]]

    remaining_categorical = [f for f in categorical_features if f not in columns["categorical"]]

    #Create New Preprocessing Pipeline
    individual_preprocessor = ColumnTransformer([("numeric", numeric_pipeline, remaining_numeric),("categorical", categorical_pipeline, remaining_categorical)])

    #Create and Train the Model
    individual_pipeline = Pipeline([("feature_engineering", FeatureEngineer()), ("preprocessor", individual_preprocessor), ("classifier", LogisticRegression(
        max_iter=1000,
        class_weight="balanced",
        random_state=42
    ))])
    individual_pipeline.fit(x_train, y_train)

    #Evaluate the Model
    predictions = individual_pipeline.predict(x_val)
    recall = recall_score(y_val, predictions, pos_label="crisis")
    precision = precision_score(y_val, predictions, pos_label="crisis")
    f1 = f1_score(y_val, predictions, pos_label="crisis")

    #Save the Results
    individual_results.append({
        "Feature Removed": feature,
        "Recall": recall,
        "Precision": precision,
        "F1 Score": f1,
        "F1 Score Change": f1 - full_model_f1
    })

#Display Results
individual_df = pd.DataFrame(individual_results)
print("Individual Feature Impact Results:")
print(individual_df.round(3).to_string(index=False))



Individual Feature Impact Results:
         Feature Removed  Recall  Precision  F1 Score  F1 Score Change
        age_at_screening   0.755      0.482     0.588           -0.010
prior_crisis_events_12mo   0.774      0.427     0.550           -0.048
              phq2_score   0.736      0.419     0.534           -0.064
 days_since_last_contact   0.774      0.466     0.582           -0.017
                  region   0.755      0.455     0.567           -0.031
          housing_status   0.698      0.451     0.548           -0.050
      substance_use_flag   0.774      0.512     0.617            0.018
    primary_support_type   0.774      0.436     0.558           -0.041


Individual Feature Impact Analysis Results:

By using the logistic regression coefficients to examine the feature relationships and performed individual feature ablation tests to measure how removing the features can affect the model performance. The results showed that
PHQ-2 score had the largest impact on the model performance and by removing it lowered the F1 score from 0.599 to 0.534. The housing status and previous crisis history also contributed important information, with the F1 score decreasing to 0.548 and 0.550 when removed. The primary support type and region also helped the model, while age and contact history had the least effects. It is interesting how removing the substance use flag increased the F1 score to 0.617, this shows us that this feature may not improve the model's predictions with what we have. Overall, the PHQ-2 score, housing status, previous crisis history and primary support type all appear to be useful features. This results shows us that some features contribute more than the others, however substance use flag may need further investigation.

Feature Importance Analysis Summary:

The feature importance analysis helped identify which features contributed most to the model's ability to predict crisis cases within the 90 day time frame. I used the logistic regression coefficients, feature group comparisons, and individual feature removal test to evaluate their impact on the model's performance.

The results showed that the PHQ-2 score had the largest impact and by removing it, it decreased the F1 score from 0.599 to 0.534. The housing status and previous crisis history were also important because when removed the F1 score decreased ro 0.548 and 0.550. Primary support type also contributed useful information, with the F1 score decreasing to 0.558. The age and contact history features had relatively little impact on the model performance, since removing them only decrease the F1 score from 0.599 to 0.588 and 0.582. This suggests that they contributed less predictive information compare to the other features in the current model.

The substance use flag was the one feature that improved the model's performance when removed. The F1 score increased from 0.599 to 0.617, precision increased from 0.488 to 0.512, and recall remained the same at 0.774. Based on these validation results, I decided to exclude the substance use flag from the current model version. This decision will be reviewed in future testing to determine whether the feature should remain excluded.

Overall, the analysis showed us that the PHQ-2 score, housing status, previous crisis history, and primary support type provided useful predictive information. These findings will help guide the future model improvements and feature selection.